# SQLiq — Ollama Backend on Google Colab

This notebook runs an **Ollama** LLM server on a Colab GPU and exposes it publicly
via **ngrok**, so your local SQLiq instance can use it as the AI backend.

**Prerequisites**
- Runtime → Change runtime type → **T4 GPU** (free tier works)
- A free [ngrok account](https://ngrok.com) — grab your auth token from the dashboard

**Model options** (pick one in Cell 5):
| Model | Size on disk | Quality | Notes |
|---|---|---|---|
| `qwen2.5-coder:7b` | ~4.7 GB | Good | Default, fits comfortably on T4 |
| `qwen2.5-coder:14b` | ~9.0 GB | **Better** ✓ | Recommended upgrade, still fits T4 |
| `codellama:13b` | ~7.4 GB | Good | Meta's code model, solid SQL |
| `qwen2.5-coder:32b` | ~19 GB | Best | Requires Colab Pro A100 |

> **Recommendation**: Use `qwen2.5-coder:14b` for noticeably better SQL generation
> with zero extra cost — it still fits on a free T4 GPU.


## Step 1 — Check GPU

In [ ]:
import subprocess

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print("⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU.")
    print(result.stderr)

## Step 2 — Install Ollama

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -5
!ollama --version

## Step 3 — Start Ollama server

In [ ]:
import subprocess
import threading
import time
import os
import requests

def _run_server():
    env = os.environ.copy()
    env["OLLAMA_ORIGINS"] = "*"          # allow requests from ngrok / any origin
    env["OLLAMA_HOST"] = "0.0.0.0:11434" # listen on all interfaces
    subprocess.run(["ollama", "serve"], env=env, capture_output=True)

server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()

# Wait until the API is ready
for attempt in range(15):
    time.sleep(1)
    try:
        r = requests.get("http://localhost:11434", timeout=2)
        if r.status_code == 200:
            print(f"✅ Ollama server ready (attempt {attempt + 1})")
            break
    except Exception:
        pass
else:
    print("❌ Ollama server did not start in time — re-run this cell")

## Step 4 — Choose and pull a model

Change `MODEL` to any option from the table above. The pull takes 2–5 minutes
depending on model size and Colab's download speed.

In [ ]:
import subprocess

# ── Pick your model ───────────────────────────────────────────────────
MODEL = "qwen2.5-coder:14b"   # change to :14b for better quality (still fits T4)
# ─────────────────────────────────────────────────────────────────────

# Skip pull if already available
existing = subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout
if MODEL.split(":")[0] in existing:
    print(f"✅ {MODEL} is already available — skipping pull")
else:
    print(f"⏳ Pulling {MODEL} (this may take a few minutes)...")
    result = subprocess.run(["ollama", "pull", MODEL], capture_output=True, text=True)
    if result.returncode == 0:
        print(f"✅ {MODEL} pulled successfully")
    else:
        print("❌ Pull failed:")
        print(result.stderr)

print("\nAvailable models:")
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)

## Step 5 — Test Ollama locally

In [ ]:
import requests, json

payload = {
    "model": MODEL,
    "messages": [{"role": "user", "content": 'Return only JSON: {"sql": "SELECT 1"}'}],
    "stream": False,
}
r = requests.post("http://localhost:11434/api/chat", json=payload, timeout=60)
r.raise_for_status()
reply = r.json()["message"]["content"]
print("Model response:", reply)

## Step 6 — Set up ngrok

1. Sign up at [ngrok.com](https://ngrok.com) (free)
2. Copy your **Authtoken** from https://dashboard.ngrok.com/get-started/your-authtoken
3. Paste it into `NGROK_TOKEN` below

In [ ]:
!pip install pyngrok -q

from pyngrok import ngrok

# ── Paste your ngrok authtoken here ──────────────────────────────────────────
# Get it from: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = "3AuKzDAHRlwBCtpbusIvlK2KeAB_nLVPTgAvcF5mmSdvjgsP"

# Optional: override with Colab Secrets (key icon in left sidebar → add NGROK_TOKEN)
try:
    from google.colab import userdata
    _secret = userdata.get("NGROK_TOKEN")
    if _secret:
        NGROK_TOKEN = _secret
        print("✅ ngrok token loaded from Colab Secrets")
    else:
        print("ℹ️  Using token from NGROK_TOKEN variable above")
except Exception:
    print("ℹ️  Using token from NGROK_TOKEN variable above")

print("✅ ngrok token ready")

## Step 7 — Open ngrok tunnel to Ollama

In [ ]:
from pyngrok import ngrok, conf

# Kill any existing tunnels and reset
ngrok.kill()

# Always set auth token here (in case cell 6 state was lost)
conf.get_default().auth_token = NGROK_TOKEN
ngrok.set_auth_token(NGROK_TOKEN)

# Expose Ollama's port
tunnel = ngrok.connect(11434, "http")
NGROK_URL = tunnel.public_url

SQLIQ_API_BASE = f"{NGROK_URL}/v1"

print(f"🌐 Tunnel URL : {NGROK_URL}")
print(f"🔗 API base   : {SQLIQ_API_BASE}")

## Step 8 — Verify Ollama through the ngrok URL

In [ ]:
import subprocess, json

# Use curl — most reliable way to test ngrok tunnels
cmd = [
    "curl", "-s", "-w", "\nHTTP_STATUS:%{http_code}",
    "-X", "POST",
    f"{SQLIQ_API_BASE}/chat/completions",
    "-H", "Content-Type: application/json",
    "-H", "ngrok-skip-browser-warning: true",
    "--max-time", "120",
    "-d", json.dumps({
        "model": MODEL,
        "messages": [{"role": "user", "content": 'Reply with only: {"status":"ok"}'}],
        "stream": False,
    }),
]

result = subprocess.run(cmd, capture_output=True, text=True)
output = result.stdout

# Split body from status line
if "\nHTTP_STATUS:" in output:
    body, status_line = output.rsplit("\nHTTP_STATUS:", 1)
    status_code = int(status_line.strip())
else:
    body, status_code = output, 0

print(f"HTTP status: {status_code}")
if status_code == 200:
    try:
        data = json.loads(body)
        print("Response:", data["choices"][0]["message"]["content"])
        print("\n✅ Ollama is reachable through ngrok")
    except Exception:
        print("Body:", body[:300])
else:
    print("Body:", body[:500] or "(empty)")
    print("\n❌ Check that Cell 6 has your real ngrok authtoken and re-run Cell 7")

## Step 9 — SQLiq configuration

Copy the output below into your local `.env` file, then run SQLiq normally with
`python main.py server`.

> **Note**: The ngrok URL changes every time you restart this notebook unless you
> have a paid ngrok plan with a static domain. Re-run Step 7 and update `.env`
> whenever you restart.

In [ ]:
print("="*60)
print("Copy these into your local .env file:")
print("="*60)
print(f"SQLIQ_API_BASE={SQLIQ_API_BASE}")
print(f"SQLIQ_MODEL={MODEL}")
print("SQLIQ_API_KEY=ollama")
print("="*60)

## Step 10 — Keep the session alive (optional)

Colab disconnects idle sessions after ~90 minutes. Run the cell below to send a
heartbeat every 60 seconds. Stop the cell manually when done.

In [ ]:
import time, requests

print("Heartbeat running — interrupt kernel (■) to stop.")
while True:
    try:
        r = requests.get("http://localhost:11434", timeout=5)
        status = "up" if r.status_code == 200 else f"status {r.status_code}"
    except Exception as e:
        status = f"error: {e}"
    print(f"[{time.strftime('%H:%M:%S')}] Ollama {status} | ngrok {NGROK_URL}", end="\r")
    time.sleep(60)